In [1]:
# !git clone https://github.com/jbloomAus/mats_sae_training.git
# %cd mats_sae_training
# !pip install -r requirements.txt

import sys
import torch
from tqdm import tqdm
from nnsight import LanguageModel

sys.path.append("./mats_sae_training")

from sae_training.sparse_autoencoder import SparseAutoencoder
from transformer_lens import utils

torch.set_grad_enabled(False)

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "jbloom/GPT2-Small-SAEs"

sae_list = []
n_layers = 12

for layer in range(n_layers):
    filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"
    resid_sae = hf_hub_download(repo_id = REPO_ID, filename = filename, local_dir="./jbloom_dictionaries")

    save_path = f"./jbloom_dictionaries/{filename}"
    sae = SparseAutoencoder.load_from_pretrained(save_path)
    sae.to("cuda:0")
    
    sae_list.append(sae)

print("Loaded dictionaries")

In [ ]:
gpt = LanguageModel("openai-community/gpt2", device_map="cuda:0", dispatch=True)

print("Loaded GPT")

In [2]:
kwargs = {
    "load_in_4bit": True,
    "device_map": "cuda:0",
    "dispatch": True
}

mixtral = LanguageModel("mistralai/Mixtral-8x7B-Instruct-v0.1", **kwargs)

print("Loaded Mixtral in 4bit")

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

Loaded Mixtral in 4bit


In [ ]:
with gpt.trace("hello, my name is"):
    activations = gpt.transformer.h[0].output[0]

    _, feature_acts, _, _, _, _ = sae_list[0](activations)
    
    feature_acts.save()

In [19]:
import simulator
import importlib 
importlib.reload(simulator)

env = simulator.Environment(mixtral, [])

In [20]:
test = simulator.Feature(
    prompt="abacdqei",
    tokens=["a","b","a","c","d","q", "e","i"], 
    acts=[],
    n_acts=["10","0","10","0","0","0","0",'0']
)

out = env(test)

print(out)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


{A}
{A}
{A}

This is because the neuron only activates for the letter 'a' and has a maximum activation value of 10. By providing multiple instances of this character, the neuron's activation will be maximized. However, the neuron does not take into account the order or combination of characters, so the specific arrangement of the letters in the sentences does not affect the activation.</s>
['A', 'A', 'A']
3
{A}
{A}
{A}

This is because the neuron only activates for the letter 'a' and has a maximum activation value of 10. By providing multiple instances of this character, the neuron's activation will be maximized. However, the neuron does not take into account the order or combination of characters, so the specific arrangement of the letters in the sentences does not affect the activation.</s>
